In [2]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, KFold, cross_validate, cross_val_predict, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
pd.set_option("display.width", 170)        
pd.set_option("display.max_columns", 20)
target   = "resolution_time_hours"          
features = ["complexity_score", "team_experience_years", "priority", "channel", "customer_tenure_months"]  
print("Target  :", target)
print("Features:", features)

Target  : resolution_time_hours
Features: ['complexity_score', 'team_experience_years', 'priority', 'channel', 'customer_tenure_months']


In [3]:
reg_df = pd.read_csv("resolution_time_regression.csv", na_values=["", " ", "NA", "N/A", "null", "?"])   
print("Shape (rows, columns):", reg_df.shape)
print()
print(reg_df.head())
print()
print(reg_df.dtypes)

Shape (rows, columns): (800, 7)

  ticket_id  complexity_score  team_experience_years priority channel  customer_tenure_months  resolution_time_hours
0     R0000               7.0                   10.0   Medium    Chat               32.638377                8.38724
1     R0001               4.0                   15.0   Medium    Chat               45.273760                0.50000
2     R0002               4.0                   13.0      Low  Portal               44.878323                0.50000
3     R0003               1.0                   10.0     High   Phone               40.095034                0.50000
4     R0004               3.0                   15.0      Low  Portal               47.451134                0.50000

ticket_id                     str
complexity_score          float64
team_experience_years     float64
priority                      str
channel                       str
customer_tenure_months    float64
resolution_time_hours     float64
dtype: object


In [4]:
print("Rows where every cell is empty   :", int(reg_df.isna().all(axis=1).sum()))
print("Columns where every cell is empty:", int(reg_df.isna().all(axis=0).sum()))
print()
missing = reg_df.isna().sum()
print(pd.DataFrame({"missing": missing,
                    "percent": (missing / len(reg_df) * 100).round(2)}))

Rows where every cell is empty   : 0
Columns where every cell is empty: 0

                        missing  percent
ticket_id                     0      0.0
complexity_score             16      2.0
team_experience_years        24      3.0
priority                      0      0.0
channel                       0      0.0
customer_tenure_months       20      2.5
resolution_time_hours         0      0.0


In [5]:
num_cols = ["complexity_score", "team_experience_years", "customer_tenure_months"]
cat_cols = ["priority", "channel"]

summary = pd.DataFrame({"mean":     reg_df[num_cols].mean(),
                        "median":   reg_df[num_cols].median(),
                        "skewness": reg_df[num_cols].skew()}).round(3)
print(summary)
print()
for col in num_cols:
    choice = "median (skewed)" if abs(summary.loc[col, "skewness"]) > 0.5 else "mean (symmetric)"
    print(f"{col:24s} -> fill blanks with the {choice}")

                          mean  median  skewness
complexity_score         5.546    6.00    -0.023
team_experience_years    7.496    8.00     0.020
customer_tenure_months  39.577   33.98     1.052

complexity_score         -> fill blanks with the mean (symmetric)
team_experience_years    -> fill blanks with the mean (symmetric)
customer_tenure_months   -> fill blanks with the median (skewed)


In [6]:
X = reg_df.drop(columns=["ticket_id", target])
y = reg_df[target]
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42)        
print("Columns used as features:", list(X.columns))
print("Train:", X_train.shape, "  Test:", X_test.shape)

Columns used as features: ['complexity_score', 'team_experience_years', 'priority', 'channel', 'customer_tenure_months']
Train: (640, 5)   Test: (160, 5)


In [7]:
train_skew = X_train[num_cols].skew()
sym_cols   = [c for c in num_cols if abs(train_skew[c]) <= 0.5]   
skew_cols  = [c for c in num_cols if abs(train_skew[c]) >  0.5]   
print("Skewness on training rows:")
print(train_skew.round(3).to_string())
print("Mean-filled  :", sym_cols)
print("Median-filled:", skew_cols)

preprocess = ColumnTransformer([
    ("num_mean",   Pipeline([("imp", SimpleImputer(strategy="mean")),
                             ("sc",  StandardScaler())]), sym_cols),
    ("num_median", Pipeline([("imp", SimpleImputer(strategy="median")),
                             ("sc",  StandardScaler())]), skew_cols),
    ("cat",        Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                             ("oh",  OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
])

def make(model):
    """Cleaning steps + a model, bundled so every score uses the same cleaning."""
    return Pipeline([("prep", preprocess), ("model", model)])
Xt = preprocess.fit_transform(X_train)
Xt = Xt.toarray() if hasattr(Xt, "toarray") else Xt
print()
print("Matrix shape after cleaning:", Xt.shape)
print("NaN values left            :", int(np.isnan(Xt).sum()))
print()
print(pd.DataFrame(Xt, columns=preprocess.get_feature_names_out()).head(3).round(2).T)

Skewness on training rows:
complexity_score         -0.015
team_experience_years     0.011
customer_tenure_months    1.000
Mean-filled  : ['complexity_score', 'team_experience_years']
Median-filled: ['customer_tenure_months']

Matrix shape after cleaning: (640, 11)
NaN values left            : 0

                                       0     1     2
num_mean__complexity_score          0.52  1.23  1.58
num_mean__team_experience_years    -0.37  0.78 -1.05
num_median__customer_tenure_months  0.35  0.70 -0.64
cat__priority_Critical              0.00  1.00  0.00
cat__priority_High                  1.00  0.00  1.00
cat__priority_Low                   0.00  0.00  0.00
cat__priority_Medium                0.00  0.00  0.00
cat__channel_Chat                   0.00  0.00  1.00
cat__channel_Email                  1.00  0.00  0.00
cat__channel_Phone                  0.00  1.00  0.00
cat__channel_Portal                 0.00  0.00  0.00


In [8]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
scoring = {"mae": "neg_mean_absolute_error", "rmse": "neg_root_mean_squared_error"}
errors = np.array([1, 3, 2])
print("MAE of [1, 3, 2] =", errors.mean())
print()
r = cross_validate(make(LinearRegression()), X_train, y_train, cv=cv, scoring=scoring)
fold_mae = -r["test_mae"]
print("MAE in each of the 5 rounds:", fold_mae.round(3))
print(f"Mean = {fold_mae.mean():.3f} hours, std = {fold_mae.std():.3f} hours")

MAE of [1, 3, 2] = 2.0

MAE in each of the 5 rounds: [2.344 1.819 1.974 2.036 2.011]
Mean = 2.037 hours, std = 0.171 hours


In [10]:
candidates = {
    "Linear Regression":       make(LinearRegression()),
    "Ridge (alpha=1)":         make(Ridge(alpha=1.0)),
    "Decision Tree (default)": make(DecisionTreeRegressor(random_state=42)),
}
rows = []
for name, pipe in candidates.items():
    r = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring)
    rows.append({"model": name, "CV MAE mean": -r["test_mae"].mean(), "CV MAE std": r["test_mae"].std()})
base = pd.DataFrame(rows).round(3)
print(base.to_string(index=False))

                  model  CV MAE mean  CV MAE std
      Linear Regression        2.037       0.171
        Ridge (alpha=1)        2.037       0.171
Decision Tree (default)        3.229       0.224


In [11]:
ridge_grid = GridSearchCV(
    make(Ridge()), {"model__alpha": [0.01, 0.1, 1, 3, 10, 30, 100, 300]},
    cv=cv, scoring="neg_mean_absolute_error")
ridge_grid.fit(X_train, y_train)

tree_grid = GridSearchCV(
    make(DecisionTreeRegressor(random_state=42)),
    {"model__max_depth": [2, 3, 4, 5, 6, 8, 10, None],
     "model__min_samples_leaf": [1, 5, 10, 20]},
    cv=cv, scoring="neg_mean_absolute_error")
tree_grid.fit(X_train, y_train)

def best_row(grid):
    i = grid.best_index_
    return -grid.cv_results_["mean_test_score"][i], grid.cv_results_["std_test_score"][i]

r_mean, r_std = best_row(ridge_grid)
t_mean, t_std = best_row(tree_grid)
print("Ridge best:", ridge_grid.best_params_, f"-> CV MAE {r_mean:.3f} +/- {r_std:.3f}")
print("Tree  best:", tree_grid.best_params_, f"-> CV MAE {t_mean:.3f} +/- {t_std:.3f}")
print()
res = pd.DataFrame(ridge_grid.cv_results_)
print(pd.DataFrame({"alpha": res["param_model__alpha"],
                    "CV MAE": -res["mean_test_score"],
                    "std": res["std_test_score"]}).round(4).to_string(index=False))

Ridge best: {'model__alpha': 1} -> CV MAE 2.037 +/- 0.171
Tree  best: {'model__max_depth': 6, 'model__min_samples_leaf': 20} -> CV MAE 2.361 +/- 0.128

 alpha  CV MAE    std
  0.01  2.0368 0.1712
  0.10  2.0368 0.1712
  1.00  2.0368 0.1710
  3.00  2.0374 0.1702
 10.00  2.0445 0.1663
 30.00  2.0804 0.1537
100.00  2.2097 0.1246
300.00  2.4297 0.1011


In [13]:
final_model = ridge_grid.best_estimator_
test_pred = final_model.predict(X_test)               
test_mae  = mean_absolute_error(y_test, test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
baseline  = mean_absolute_error(y_test, np.full(len(y_test), y_train.median()))

print("Selected model : Ridge, alpha =", ridge_grid.best_params_["model__alpha"])
print(f"Test MAE       : {test_mae:.3f} hours")
print(f"Test RMSE      : {test_rmse:.3f} hours")
print(f"Baseline MAE   : {baseline:.3f} hours ")

Selected model : Ridge, alpha = 1
Test MAE       : 2.228 hours
Test RMSE      : 3.400 hours
Baseline MAE   : 3.273 hours 


In [14]:
y_all = reg_df[target]
q1, q3 = y_all.quantile([0.25, 0.75])
iqr    = q3 - q1
upper  = q3 + 1.5 * iqr
lower  = q1 - 1.5 * iqr
is_out = (y_all > upper) | (y_all < lower)

print(f"Q1 = {q1:.2f}, Q3 = {q3:.2f}, fences = [{lower:.2f}, {upper:.2f}]")
print(f"Outliers: {int(is_out.sum())} of {len(y_all)} ({is_out.mean()*100:.1f}%), longest = {y_all.max():.2f} h")
print("Outliers by priority:", reg_df.loc[is_out, "priority"].value_counts().to_dict())
print(f"Mean = {y_all.mean():.2f}, median = {y_all.median():.2f}, skewness = {y_all.skew():.2f}")
print("Tickets at exactly 0.5 hours      :", int((y_all == 0.5).sum()))
print("Customers at exactly 120 months   :", int((reg_df["customer_tenure_months"] == 120).sum()))

Q1 = 2.63, Q3 = 8.35, fences = [-5.94, 16.92]
Outliers: 10 of 800 (1.2%), longest = 27.49 h
Outliers by priority: {'High': 3, 'Low': 3, 'Medium': 3, 'Critical': 1}
Mean = 5.87, median = 5.39, skewness = 1.18
Tickets at exactly 0.5 hours      : 72
Customers at exactly 120 months   : 15


In [15]:
rows = []
models = {"Linear Regression":    make(LinearRegression()),
          "Ridge (tuned)":        ridge_grid.best_estimator_,
          "Decision Tree (tuned)": tree_grid.best_estimator_}
for name, pipe in models.items():
    pred = cross_val_predict(pipe, X_train, y_train, cv=cv)   
    e    = y_train.values - pred
    sq   = np.sort(e ** 2)[::-1]
    rows.append({"model": name,
                 "MAE": np.abs(e).mean(),
                 "RMSE": np.sqrt((e ** 2).mean()),
                 "worst 5% share of squared error": sq[: int(0.05 * len(e))].sum() / sq.sum()})
print(pd.DataFrame(rows).round(3).to_string(index=False))
A = np.array([1] * 9 + [10])      
B = np.full(10, 2.5)        
print()
for name, e in [("A", A), ("B", B)]:
    print(f"Model {name}: MAE = {e.mean():.2f}, RMSE = {np.sqrt((e ** 2).mean()):.2f}")

                model   MAE  RMSE  worst 5% share of squared error
    Linear Regression 2.037 3.019                            0.555
        Ridge (tuned) 2.037 3.018                            0.555
Decision Tree (tuned) 2.361 3.317                            0.479

Model A: MAE = 1.90, RMSE = 3.30
Model B: MAE = 2.50, RMSE = 2.50


In [16]:
lin   = make(LinearRegression()).fit(X_train, y_train)
names = lin.named_steps["prep"].get_feature_names_out()
coefs = pd.Series(lin.named_steps["model"].coef_, index=names).round(2)
print("Change in predicted hours for a one-step rise in each feature:")
print(coefs.to_string())

Change in predicted hours for a one-step rise in each feature:
num_mean__complexity_score            2.16
num_mean__team_experience_years      -1.05
num_median__customer_tenure_months   -0.55
cat__priority_Critical                2.90
cat__priority_High                   -0.04
cat__priority_Low                    -1.35
cat__priority_Medium                 -1.51
cat__channel_Chat                    -0.42
cat__channel_Email                    0.59
cat__channel_Phone                   -0.11
cat__channel_Portal                  -0.05
